# 01 — The store: what `monad_wordnet_v4_20260519.bin` contained
**Scope.** This notebook only *reads* the PTOL state file that the May-2026 C monad (`PtolC/` at commit `eb28ce8`, v1.215) loaded when it produced the
screenshot reply `holcussamestuffseemy`. Nothing is modified, tuned or sanitised. Reader: `tools/ptol_bin.py`, a copy of the layout in `state.c: state_load`.

| section | bytes | meaning |
|---|---|---|
| header | 36 (v4) | `"PTOL"`, version, N, vocab_size, A_size, word_count, threshold f64, affect f32 |
| β | 8·N | depth per zero slot (f64) |
| age | 4·N | steps since last activation (i32) |
| vocab | variable | per *occupied slot*: `idx u32, wlen u16, E f64, home u8, gen u8, prose_seen u8, word` |
| A | 16·A_size | `i u32, j u32, w f64`, always `i < j` |

In [1]:
import os, sys, json, math, collections, re, hashlib
import numpy as np
sys.path.insert(0, os.path.abspath("../tools"))
import ptol_bin, monad_py as mp
ROOT  = os.path.abspath("..")
STORE = os.path.join(ROOT, "dirty", "monad_wordnet_v4_20260519.bin")
CBIN  = os.path.join(ROOT, "build", "PtolC_eb28ce8", "ptolemy")
SPACES = "[\u200b\u2009\u2008\u2007\u2006\u2005\u2004\u2003]"   # the eight Fermat separators

In [2]:
d = ptol_bin.load(STORE)
print({k: d[k] for k in ("version","N","vocab_size","A_size","word_count","threshold","affect")})
print("file bytes", d["bytes_total"], " consumed by the reader", d["bytes_consumed"], " -> every byte accounted for:", d["bytes_total"]==d["bytes_consumed"])
print("sha256", hashlib.sha256(open(STORE,"rb").read()).hexdigest())

{'version': 4, 'N': 25000, 'vocab_size': 24485, 'A_size': 6825748, 'word_count': 121914388, 'threshold': 3.776, 'affect': -1.0}
file bytes 110040527  consumed by the reader 110040527  -> every byte accounted for: True


sha256 66f70c70dcd8d860fa4af01833cf5c6e4c376194c510b5be156fe0c6064faf31


The header matches the three numbers printed on the screenshot (`vocab=24485  A=6825748  wc=121914388`), so this is the file that was loaded.

## What a "vocabulary" entry is
The store has **one word per occupied slot** (25 000 slots), not one per word it has met. Which word a slot displays is decided in `monad_learn_ex`:
`if (!present || E > vocab[idx].E) replace` — the arrival with the highest `E` wins. (Notebook 02 shows `E` is a function of spelling alone.)

In [3]:
V = d["vocab"]; words = [v[1] for v in V]
print("occupied slots", len(V), "of", d["N"])
print("word length distribution of the displayed words:", sorted(collections.Counter(len(w) for w in words).items()))
print("non-ASCII words:", sum(1 for w in words if not w.isascii()))

occupied slots 24485 of 25000
word length distribution of the displayed words: [(2, 200), (3, 2518), (4, 6875), (5, 12545), (6, 2330), (7, 17)]
non-ASCII words: 0


## Scripts: is there any multilingual content?

In [4]:
A = json.load(open(os.path.join(ROOT,"dirty","monad_wordnet_v4_20260519.assessment.json")))
print(json.dumps(A["scripts"], ensure_ascii=False, indent=1)[:600])

{
 "word_scripts": {
  "ASCII": 24485
 },
 "nonascii_chars": {},
 "nonascii_word_count": 0,
 "nonascii_samples": []
}


All 24 485 displayed words are ASCII. Notebook 03 shows why none could have been anything else: `tok_split` uses `isalpha()` in the C locale, so a non-ASCII
byte is a *separator*, never part of a word.

## Field health (the project's own `eval_checkpoint.py` measures)

In [5]:
print(json.dumps({k: A[k] for k in ("field","eval_checkpoint")}, indent=1))

{
 "field": {
  "occupied_zeros": 25000,
  "entropy": 12.9558,
  "entropy_max": 14.6096,
  "entropy_pct": 88.68,
  "gini": 0.7564,
  "beta_saturated": 1826,
  "beta_max": 7.552
 },
 "eval_checkpoint": {
  "polluted": 1233,
  "clean": 23252,
  "pollution_pct": 5.04,
  "by_category": {
   "consonant-only": 993,
   "consonant-cluster-3": 654,
   "no-vowel": 344,
   "hex-string": 268
  }
 }
}


`eval_checkpoint.py` verdict rule: CONVERGED = pollution < 1 % and entropy and Gini in band; PASS = pollution < 20 %. Here: pollution 5.04 %, entropy 88.68 % of max
(natural band 85–92), Gini 0.7564 (natural band 0.60–0.80). → **PASS, not CONVERGED.**

## Token sanitiser (2026-10-17 ruleset) — same tool that wrote the `*.POISON.md` notes

In [6]:
for k, n in list(A["sanitiser"]["tiers"].items())[:10]:
    print(f"{k:34} {n:6}  {A['sanitiser']['samples'][k][:8]}")
print("quarantined:", A["sanitiser"]["quarantined"], f"({A['sanitiser']['quarantined_pct']} %)")

B:candidate_evidence_pending        18409  ['seemy', 'carre', 'azv', 'xxa', 'diem', 'gaii', 'uxp', 'arcos']
A:in_wordnet                         4563  ['fluid', 'papaw', 'pommy', 'quiets', 'farm', 'atrip', 'peaty', 'citric']
Q:no_vowel                            990  ['tcpd', 'vqr', 'zbk', 'db', 'hgc', 'fqb', 'plb', 'btlq']
C:proper_noun                         450  ['aiai', 'frey', 'donar', 'ashton', 'stalin', 'speke', 'beria', 'delhi']
F:function_word                        55  ["i's", "you'd", 'same', 'then', 'upon', 'once', 'while', 'than']
Q:chrome                               14  ['jstor', 'the', 'this', 'oclc', 'log', 'edit', 'export', 'doi']
Q:single_letter                         3  ["p's", "w's", "b's"]
Q:non_latin_or_symbol                   1  ["s''v"]
quarantined: 1008 (4.12 %)


`B:candidate_evidence_pending` is "not found in WordNet" — 75 % of the displayed words. Most of those are *slot winners that are not words*
(`azv`, `xxa`, `uxp`, …): see notebook 06, item 2.

## Where the screenshot's words and the prompt's words sit

In [7]:
for w, hit in A["probes"].items(): print(f"{w:13}", hit if hit else "— not displayed by any slot")

holcus        [{'idx': 24639, 'E': 0.5625, 'beta': 7.552, 'age': 9, 'home': 1, 'gen': 1, 'prose_seen': 3}]
philadelphos  — not displayed by any slot
same          [{'idx': 1859, 'E': 0.2699, 'beta': 7.552, 'age': 9, 'home': 1, 'gen': 1, 'prose_seen': 3}]
stuff         [{'idx': 18482, 'E': 0.4834, 'beta': 7.552, 'age': 9, 'home': 1, 'gen': 1, 'prose_seen': 3}]
seemy         [{'idx': 0, 'E': 0.246, 'beta': 7.552, 'age': 0, 'home': 1, 'gen': 1, 'prose_seen': 3}]
what          [{'idx': 16194, 'E': 0.454, 'beta': 7.552, 'age': 0, 'home': 1, 'gen': 1, 'prose_seen': 3}]
is            — not displayed by any slot
your          [{'idx': 18893, 'E': 0.4887, 'beta': 7.552, 'age': 9, 'home': 1, 'gen': 1, 'prose_seen': 3}]
name          — not displayed by any slot
tree          — not displayed by any slot
water         — not displayed by any slot
setn          [{'idx': 16406, 'E': 0.4568, 'beta': 7.552, 'age': 9, 'home': 1, 'gen': 1, 'prose_seen': 3}]
abysmal       [{'idx': 16796, 'E': 0.4618, 'beta

`tree`, `water`, `name`, `is` are **not** displayed anywhere, because the slot each of them addresses to is held by a different word:

In [8]:
for w in ["tree","water","name","is","ptolemy","philadelphos"]:
    i, E = mp.word_coords(w); print(f"{w:13} -> z#{i:<6} E={E:.4f}   slot displays: {dict(zip((v[0] for v in V),(v[1] for v in V))).get(i)!r}")

tree          -> z#3109   E=0.2859   slot displays: 'typhon'
water         -> z#9361   E=0.3663   slot displays: 'defrg'
name          -> z#15471  E=0.4447   slot displays: 'emite'
is            -> z#1894   E=0.2703   slot displays: 'nanti'
ptolemy       -> z#10156  E=0.3765   slot displays: 'sift'
philadelphos  -> z#0      E=0.2460   slot displays: 'seemy'
